# 04 — Make manuscript and supplementary figures

Consolidated publication figure notebook using outputs from notebooks 01–03.


## 0. Packages, paths and plotting theme


In [ ]:
library(mgcv)
library(ggplot2)
library(terra)
library(tidyterra)
library(sf)
library(rnaturalearth)
library(rnaturalearthdata)
library(data.table)
library(cowplot)
library(grid)

fig <- function(height, width) {
  options(repr.plot.width = width, repr.plot.height = height)
}

plot_theme1 <- theme(
  strip.text = element_text(size = 16, face = "bold"),
  strip.background = element_blank(),
  axis.text = element_text(size = 18, colour = "black"),
  axis.title = element_text(size = 22),
  panel.grid = element_blank(),
  legend.position = "bottom",
  legend.title = element_text(size = 16, face = "bold"),
  legend.text = element_text(size = 14)
)

find_project_root <- function(start = getwd()) {
  p <- normalizePath(start, mustWork = TRUE)
  candidates <- unique(c(p, dirname(p), dirname(dirname(p))))

  for (candidate in candidates) {
    if (dir.exists(file.path(candidate, "data")) &&
        dir.exists(file.path(candidate, "src"))) {
      return(normalizePath(candidate, mustWork = TRUE))
    }
  }

  stop("Could not locate publication repository root.")
}

project_dir <- find_project_root()

data_dir <- file.path(project_dir, "data")
figmaps_dir <- file.path(project_dir, "outputs", "figure_maps")
driver_dir <- file.path(project_dir, "outputs", "driver_maps")
figure_data_dir <- file.path(project_dir, "outputs", "figure_data")
figure_out_dir <- file.path(project_dir, "outputs", "manuscript_figures")

dir.create(figure_out_dir, recursive = TRUE, showWarnings = FALSE)

world <- ne_countries(scale = "large", returnclass = "sf")

cat("Project:", project_dir, "\n")
cat("Figure data:", figure_data_dir, "\n")
cat("Figure output:", figure_out_dir, "\n")


## 1. Temporal calibration GAM

This is the current production calibration, not the older exploratory GAM material. It rebuilds `gam_age_curve.csv` and the manuscript calibration figure from the repaired chronology dataset.


In [ ]:
input_file <- file.path(
  data_dir, "calibration", "wreck_calibration.csv"
)
curve_file <- file.path(data_dir, "gam_age_curve.csv")

if (!file.exists(input_file)) stop("Missing calibration input: ", input_file)
if (!dir.exists(data_dir)) dir.create(data_dir, recursive = TRUE)


### 1.1 Calibration subset


In [ ]:
df <- read.csv(input_file)

required <- c(
  "Obs",
  "chronological_age",
  "log_CR_physics",
  "log_CR_bio"
)

missing <- setdiff(required, names(df))
if (length(missing) > 0) {
  stop("Missing required columns: ", paste(missing, collapse = ", "))
}

cal <- df[
  complete.cases(df[, required]) &
    df$Obs > 0 &
    df$chronological_age > 0,
]

cal$Obs <- as.integer(cal$Obs)

cat("Rows in public calibration file:", nrow(df), "\n")
cat("Rows used for GAM calibration:", nrow(cal), "\n")
cat("Age range:", paste(range(cal$chronological_age), collapse = " to "), "\n")
cat("Observed classes:", paste(sort(unique(cal$Obs)), collapse = ", "), "\n")

expected_classes <- seq_len(length(unique(cal$Obs)))
if (!identical(sort(unique(cal$Obs)), expected_classes)) {
  stop("Observed Muckelroy classes are not consecutive integers starting at 1.")
}


### 1.2 Fit calibration models


In [ ]:
Rcat <- length(unique(cal$Obs))

model1 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5) +
    s(log_CR_bio, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model2 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model4 <- gam(
  Obs ~ s(chronological_age, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

summary(model1)
AIC(model1, model2, model4)

gam.check(model1)
gam.check(model2)
gam.check(model4)


### 1.3 Compare age–condition functions


In [ ]:
age_compare <- seq(
  min(cal$chronological_age),
  max(cal$chronological_age),
  length.out = 300
)

mean_phys <- mean(cal$log_CR_physics, na.rm = TRUE)
mean_bio <- mean(cal$log_CR_bio, na.rm = TRUE)

new_m1 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

new_m2 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys
)

new_m4 <- data.frame(
  chronological_age = age_compare
)

expected_class <- function(model, newdata) {
  p <- predict(model, newdata = newdata, type = "response")
  as.numeric(p %*% seq_len(ncol(p)))
}

e1 <- expected_class(model1, new_m1)
e2 <- expected_class(model2, new_m2)
e4 <- expected_class(model4, new_m4)

fig(6, 8)
plot(
  age_compare, e4,
  type = "l",
  lwd = 2,
  ylim = range(c(e1, e2, e4)),
  xlab = "Chronological age (years)",
  ylab = "Expected Muckelroy condition class"
)

lines(age_compare, e2, lwd = 2, lty = 2)
lines(age_compare, e1, lwd = 2, lty = 3)

legend(
  "bottomright",
  legend = c(
    "Age only",
    "Age + physical CR",
    "Age + physical + biological CR"
  ),
  lty = c(1, 2, 3),
  lwd = 2,
  bty = "n"
)


### 1.4 Empirical support


In [ ]:
age_quantiles <- quantile(
  cal$chronological_age,
  probs = c(0.10, 0.90, 0.95, 0.99)
)

print(age_quantiles)

support_age_max <- 125
tail_lambda0 <- 0.01

cat("Production support age:", support_age_max, "years\n")
cat("Wrecks older than support age:",
    sum(cal$chronological_age > support_age_max), "\n")


### 1.4a GAM basis / k comparison


In [ ]:
# Sensitivity of the age smooth to spline basis and k
age_basis_grid <- seq(
  min(cal$chronological_age, na.rm = TRUE),
  max(cal$chronological_age, na.rm = TRUE),
  length.out = 200
)

basis_newdata <- data.frame(
  chronological_age = age_basis_grid,
  log_CR_physics = mean(cal$log_CR_physics, na.rm = TRUE),
  log_CR_bio = mean(cal$log_CR_bio, na.rm = TRUE)
)

fit_basis_model <- function(bs, k) {
  gam(
    Obs ~
      s(chronological_age, k = k, bs = bs) +
      s(log_CR_physics, k = 5) +
      s(log_CR_bio, k = 5),
    family = ocat(R = Rcat),
    data = cal,
    method = "REML"
  )
}

basis_settings <- expand.grid(
  bs = c("tp", "cr"),
  k = c(5, 7),
  stringsAsFactors = FALSE
)

basis_pred_list <- vector("list", nrow(basis_settings))

for (i in seq_len(nrow(basis_settings))) {
  m <- fit_basis_model(basis_settings$bs[i], basis_settings$k[i])

  basis_pred_list[[i]] <- data.frame(
    chronological_age = age_basis_grid,
    expected_class = expected_class(m, basis_newdata),
    basis = basis_settings$bs[i],
    k = basis_settings$k[i],
    label = paste0(basis_settings$bs[i], ", k = ", basis_settings$k[i])
  )
}

basis_pred_df <- do.call(rbind, basis_pred_list)

p_basis <- ggplot(
  basis_pred_df,
  aes(
    x = chronological_age,
    y = expected_class,
    linetype = label,
    linewidth = label
  )
) +
  geom_line() +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  scale_linetype_manual(
    values = c(
      "tp, k = 5" = "solid",
      "tp, k = 7" = "longdash",
      "cr, k = 5" = "dotted",
      "cr, k = 7" = "dotdash"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "tp, k = 5" = 1.3,
      "tp, k = 7" = 0.8,
      "cr, k = 5" = 1.3,
      "cr, k = 7" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Chronological age (years)",
    y = "Expected condition class",
    linetype = "Basis / k"
  ) +
  theme_classic() +
  plot_theme1

p_basis


### 1.4b Chronological age by Muckelroy class


In [ ]:
p_age_class <- ggplot(
  cal,
  aes(x = as.factor(Obs), y = chronological_age)
) +
  geom_boxplot(
    alpha = 0.4,
    outlier.shape = NA
  ) +
  geom_jitter(
    width = 0.2,
    height = 0,
    size = 2,
    alpha = 0.7,
    color = "black"
  ) +
  labs(
    x = "Muckelroy class",
    y = "Age"
  ) +
  theme_classic() +
  plot_theme1

p_age_class


### 1.4c Age distribution of calibration wrecks


In [ ]:
# Supplementary wreck-count distribution.
# support_age_max marks the 125-year empirical support threshold used by the model.
p_wreck_counts <- ggplot(
  cal,
  aes(x = chronological_age)
) +
  geom_histogram(
    bins = 10,
    fill = "grey80",
    color = "black"
  ) +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  scale_y_continuous(expand = c(0, 0)) +
  labs(
    x = "Chronological age (years)",
    y = "Wreck count"
  ) +
  theme_classic() +
  plot_theme1

p_wreck_counts


### 1.5 Production age curve


In [ ]:
age_grid <- seq(0, 300, by = 1)

newdata <- data.frame(
  chronological_age = age_grid,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

raw_pred <- expected_class(model1, newdata)

mono_pred <- cummax(raw_pred)
model_pred <- mono_pred

support_idx <- which(age_grid <= support_age_max)
tail_idx <- which(age_grid > support_age_max)

class_at_support <- mono_pred[max(support_idx)]

model_pred[tail_idx] <- 4 - (4 - class_at_support) *
  exp(-tail_lambda0 * (age_grid[tail_idx] - support_age_max))

model_pred <- pmin(model_pred, 4)

gam_curve_out <- data.frame(
  Age = age_grid,
  Raw_GAM = raw_pred,
  Monotonic_GAM = mono_pred,
  ExpectedClass = model_pred
)

if (nrow(gam_curve_out) != 301) stop("Unexpected gam_age_curve row count.")
if (!identical(names(gam_curve_out),
               c("Age", "Raw_GAM", "Monotonic_GAM", "ExpectedClass"))) {
  stop("Unexpected gam_age_curve column structure.")
}
if (any(diff(gam_curve_out$ExpectedClass) < 0)) {
  stop("Production ExpectedClass curve is not monotonic.")
}

write.csv(
  gam_curve_out,
  file = curve_file,
  row.names = FALSE
)

cat("Saved:", curve_file, "\n")
cat("Expected class at 125 years:",
    gam_curve_out$ExpectedClass[gam_curve_out$Age == 125], "\n")

head(gam_curve_out)
tail(gam_curve_out)


### 1.6 Bootstrap uncertainty


In [ ]:
figure_age <- seq(
  min(cal$chronological_age, na.rm = TRUE),
  250,
  by = 1
)

figure_newdata <- data.frame(
  chronological_age = figure_age,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

figure_raw <- expected_class(model1, figure_newdata)
figure_mono <- cummax(figure_raw)
figure_model <- figure_mono

support_idx_fig <- which(figure_age <= support_age_max)
tail_idx_fig <- which(figure_age > support_age_max)

class_at_support_fig <- figure_mono[max(support_idx_fig)]

figure_model[tail_idx_fig] <- 4 - (4 - class_at_support_fig) *
  exp(-tail_lambda0 * (figure_age[tail_idx_fig] - support_age_max))

figure_model <- pmin(figure_model, 4)

curve_df <- data.frame(
  Age = figure_age,
  Raw_GAM = figure_raw,
  Monotonic_GAM = figure_mono,
  Model_curve = figure_model
)

set.seed(123)

B <- 300
pred_mat <- matrix(
  NA_real_,
  nrow = length(figure_age),
  ncol = B
)

for (b in seq_len(B)) {
  idx <- sample(seq_len(nrow(cal)), replace = TRUE)
  dat_b <- cal[idx, ]

  fit_b <- gam(
    Obs ~
      s(chronological_age, k = 5) +
      s(log_CR_physics, k = 5) +
      s(log_CR_bio, k = 5),
    family = ocat(R = Rcat),
    data = dat_b,
    method = "REML"
  )

  pred_mat[, b] <- expected_class(fit_b, figure_newdata)
}

curve_df$Raw_GAM_lwr <- apply(
  pred_mat, 1, quantile, 0.025, na.rm = TRUE
)
curve_df$Raw_GAM_upr <- apply(
  pred_mat, 1, quantile, 0.975, na.rm = TRUE
)

raw_plot_df <- curve_df[
  curve_df$Age <= max(cal$chronological_age, na.rm = TRUE),
]


### 1.7 Observed versus condition-derived chronological age


In [ ]:
# Apparent chronological age inferred from observed Muckelroy condition class
age_lookup <- curve_df[, c("Age", "Model_curve")]

cal$predicted_chronological_age <- NA_real_

for (i in seq_len(nrow(cal))) {
  obs_i <- cal$Obs[i]

  if (!is.na(obs_i)) {
    j <- which.min(abs(age_lookup$Model_curve - obs_i))
    cal$predicted_chronological_age[i] <- age_lookup$Age[j]
  }
}

p_age_validation <- ggplot(
  cal,
  aes(
    x = chronological_age,
    y = predicted_chronological_age
  )
) +
  geom_point(
    alpha = 0.45,
    color = "black",
    size = 2
  ) +
  geom_abline(
    intercept = 0,
    slope = 1,
    linetype = "dotted"
  ) +
  geom_smooth(
    method = "lm",
    formula = y ~ x,
    se = TRUE,
    color = "black",
    fill = "grey70",
    linewidth = 1.2
  ) +
  labs(
    x = "Observed chronological age (years)",
    y = "Predicted chronological age (years)"
  ) +
  theme_classic() +
  plot_theme1

p_age_validation


### 1.8 Manuscript calibration figure


In [ ]:
#fig(7, 10)

final_figure <- ggplot() +
#  geom_point(
#    data = cal,
#    aes(x = chronological_age, y = Obs),
#    alpha = 0.45,
#    color = "blue",
#    size = 2
#  ) +
  geom_ribbon(
    data = raw_plot_df,
    aes(
      x = Age,
      ymin = Raw_GAM_lwr,
      ymax = Raw_GAM_upr
    ),
    alpha = 0.2
  ) +
geom_jitter(
    data = cal,
    aes(x = chronological_age, y = Obs),
    width = 0,
    height = 0.06,
    alpha = 0.45,
    color = "black",
    size = 2
  ) +
  geom_line(
    data = raw_plot_df,
    aes(x = Age, y = Raw_GAM),
    linewidth = 1,
    linetype = "dashed"
  ) +
  geom_line(
    data = curve_df,
    aes(x = Age, y = Model_curve),
    linewidth = 1.4
  ) +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  geom_rug(
    data = cal,
    aes(x = chronological_age, y = Obs),
    sides = "b"
  ) +
  coord_cartesian(ylim = c(0.5, 4.1)) +
  xlab("Chronological age (years)") +
  ylab("Muckelroy class") +
  theme_classic() +
  theme(
    plot.title = element_blank(),
    axis.text = element_text(size = 14),
    axis.title = element_text(size = 14),
    panel.background = element_blank(),
    panel.grid = element_blank(),
    axis.line = element_line(colour = "black")
  )

final_figure


In [ ]:
ggsave(
  file.path(figure_out_dir, "gam_age_condition_curve.png"),
  final_figure,
  width = 10, height = 7, dpi = 300, bg = "white"
)


ggsave(
  file.path(figure_out_dir, "gam_basis_k_comparison.png"),
  p_basis,
  width = 9, height = 7, dpi = 300, bg = "white"
)

ggsave(
  file.path(figure_out_dir, "chronological_age_by_muckelroy_class.png"),
  p_age_class,
  width = 8, height = 7, dpi = 300, bg = "white"
)

ggsave(
  file.path(figure_out_dir, "wreck_age_distribution.png"),
  p_wreck_counts,
  width = 8, height = 7, dpi = 300, bg = "white"
)



ggsave(
  file.path(figure_out_dir, "observed_vs_predicted_chronological_age.png"),
  p_age_validation,
  width = 8, height = 8, dpi = 300, bg = "white"
)


## 2. Regional time to Muckelroy class 3


In [ ]:
steel <- rast(file.path(figmaps_dir, "steel_none_time_to_muckelroy_3.tif"))
woodnone <- rast(file.path(figmaps_dir, "wood_none_time_to_muckelroy_3.tif"))
woodhigh <- rast(file.path(figmaps_dir, "wood_high_time_to_muckelroy_3.tif"))

threshold_limits <- range(
  c(values(steel), values(woodnone), values(woodhigh)),
  na.rm = TRUE
)

r <- steel

map_threshold <- function(x, title) {
  ggplot(world) +
    theme_bw() +
    labs(title = title) +
    geom_spatraster(data = x) +
    geom_sf() +
    coord_sf(
      xlim = c(xmin(r), xmax(r)),
      ylim = c(ymin(r), ymax(r)),
      expand = FALSE
    ) +
    scale_fill_viridis_c(
      name = "Years to\nMuckelroy 3\n",
      limits = threshold_limits,
      oob = scales::squish
    ) +
    theme(
      axis.text = element_text(size = 16, colour = "black"),
      panel.background = element_rect(fill = "lightblue1", colour = "lightblue1"),
      panel.grid.major = element_line(linewidth = 0.5, linetype = "solid", colour = "grey"),
      plot.title = element_text(hjust = 0, face = "bold", size = 22),
      legend.title = element_text(size = 12),
      legend.text = element_text(size = 10),
      plot.margin = margin(8, 5, 8, 5)
    )
}

steelplot <- map_threshold(steel, "a) Steel wrecks")
woodplot <- map_threshold(woodnone, "b) Wood wrecks, no Teredo")
teredoplot <- map_threshold(woodhigh, "c) Wood wrecks, high Teredo")

fig(15, 12)
threshold_maps <- cowplot::plot_grid(
  steelplot, woodplot, teredoplot,
  ncol = 1,
  align = "v",
  axis = "lr"
)
threshold_maps


## 3. Environmental and mechanical-forcing drivers


In [ ]:
sal <- rast(file.path(driver_dir, "salinity_mean.tif"))
temp <- rast(file.path(driver_dir, "bottom_temperature_mean.tif"))
mechanical <- rast(file.path(driver_dir, "mechanical_forcing_modifier.tif"))

sites <- data.frame(
  site = c(
    "Baltic refuge", "Danish Straits", "Jutland shelf",
    "Southern North Sea", "Norwegian trench", "Baltic entrance"
  ),
  latitude = c(56.5, 56.3, 56.4, 54.8, 57.35, 55),
  longitude = c(18.0, 11.5, 6.4, 6.5, 7.7, 13)
)

sites_sf <- st_as_sf(
  sites,
  coords = c("longitude", "latitude"),
  crs = 4326
)

base_driver_map <- function(x) {
  ggplot(world) +
    theme_bw() +
    geom_spatraster(data = x) +
    geom_sf() +
    geom_sf(
      data = sites_sf,
      colour = "black", fill = "black",
      shape = 21, size = 3
    ) +
    coord_sf(
      xlim = c(xmin(r), xmax(r)),
      ylim = c(ymin(r), ymax(r)),
      expand = FALSE
    ) +
    theme(
      axis.text = element_text(size = 16, colour = "black"),
      panel.background = element_rect(fill = "lightblue1", colour = "lightblue1"),
      panel.grid.major = element_line(linewidth = 0.5, linetype = "solid", colour = "grey")
    )
}

templot <- base_driver_map(temp) +
  scale_fill_gradientn(
    colours = c("#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"),
    name = "Temperature\n(°C)\n"
  ) +
  labs(title = "a) Seafloor temperature")

salplot <- base_driver_map(sal) +
  scale_fill_distiller(
    palette = "RdYlBu",
    direction = 1,
    name = "Salinity\n(PSU)\n"
  ) +
  labs(title = "b) Salinity")

hydroplot <- base_driver_map(mechanical) +
  scale_fill_gradientn(
    colours = c("white", "lightblue", "royalblue", "navy"),
    name = "Forcing\nstrength\n",
    limits = c(0, 1),
    oob = scales::squish
  ) +
  labs(title = "c) Mechanical forcing")

driver_title_theme <- theme(
  plot.title = element_text(hjust = 0, face = "bold", size = 22),
  legend.title = element_text(size = 12),
  legend.text = element_text(size = 10),
  plot.margin = margin(8, 5, 8, 5)
)

templot <- templot + driver_title_theme
salplot <- salplot + driver_title_theme
hydroplot <- hydroplot + driver_title_theme

fig(15, 12)
driver_maps <- cowplot::plot_grid(
  templot, salplot, hydroplot,
  ncol = 1,
  align = "v",
  axis = "lr"
)
driver_maps


## 4. Representative condition trajectories


In [ ]:
traj <- fread(file.path(
  figure_data_dir,
  "representative_site_trajectories.csv"
))

traj[, site := factor(
  site,
  levels = c(
    "Norwegian trench",
    "Danish Straits",
    "Jutland shelf",
    "Baltic entrance",
    "Southern North Sea",
    "Baltic"
  )
)]

traj[, scenario := factor(
  scenario,
  levels = c("Steel", "Wood_none", "Wood_high"),
  labels = c("Steel", "Wood, No Teredo", "Wood, High Teredo")
)]

p_traj <- ggplot(
  traj,
  aes(
    x = chronological_age,
    y = predicted_condition,
    linetype = scenario
  )
) +
  geom_line(linewidth = 1.2) +
  scale_linetype_manual(
    values = c(
      "Steel" = "solid",
      "Wood, No Teredo" = "longdash",
      "Wood, High Teredo" = "dotted"
    )
  ) +
  facet_wrap(~ site, ncol = 2) +
  geom_hline(yintercept = 3, linetype = "dotted") +
  coord_cartesian(ylim = c(1, 4), xlim = c(0, 300)) +
  labs(
    x = "Chronological age (years)",
    y = "Predicted Muckelroy condition class",
    linetype = "Scenario"
  ) +
  theme_classic() +
  plot_theme1

fig(10, 12)
p_traj


## 5. Sensitivity figures

The sensitivity analysis itself remains in its dedicated notebook. These cells only read the final exported summaries and make the manuscript tornado and supplementary cell-ratio figure.


In [ ]:
parameter_labels <- c(
  Ea = "Activation energy",
  U_current_ref = "Current reference velocity",
  U_wave_ref = "Wave reference velocity",
  hydro_interaction_strength = "Current–wave interaction",
  S_mid_corr = "Salinity midpoint",
  burial_alpha = "Burial exponent",
  burial_strength = "Burial strength",
  k_teredo_high = "Teredo intensity"
)

tornado <- fread(file.path(
  figure_data_dir,
  "fig5_tornado_summary.csv"
))

tornado[, label := parameter_labels[parameter]]
tornado[, low_pct := 100 * (low - 1)]
tornado[, high_pct := 100 * (high - 1)]
tornado[, max_abs_pct := pmax(abs(low_pct), abs(high_pct))]

tornado <- tornado[order(max_abs_pct)]
tornado[, label := factor(label, levels = label)]

p_sensitivity <- ggplot(tornado, aes(y = label)) +
  geom_segment(
    aes(x = low_pct, xend = high_pct, yend = label),
    linewidth = 5,
    lineend = "butt"
  ) +
  geom_vline(xintercept = 0, linetype = "dotted") +
  labs(
    x = "Change from baseline median rate scale (%)",
    y = NULL
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_sensitivity


In [ ]:
cell <- fread(file.path(
  figure_data_dir,
  "supp_sensitivity_cell_ratios.csv"
))

cell[, value_plot := value]
cell[parameter == "Ea", value_plot := value_plot / 1000]

label_order <- c(
  "Current reference velocity",
  "Activation energy",
  "Teredo intensity",
  "Burial strength",
  "Burial exponent",
  "Wave reference velocity",
  "Salinity midpoint",
  "Current–wave interaction"
)

cell[, label := factor(
  parameter_labels[parameter],
  levels = label_order
)]

# Factor separately within the combined data. Free x scales handle parameter-specific values.
cell[, value_plot := factor(value_plot, levels = sort(unique(value_plot)))]

p_supp_sensitivity <- ggplot(cell) +
  geom_boxplot(
    aes(x = value_plot, y = ratio),
    outlier.shape = NA,
    width = 0.65
  ) +
  geom_hline(
    yintercept = 1,
    linetype = "dashed",
    linewidth = 0.4
  ) +
  facet_wrap(
    ~ label,
    scales = "free_x",
    ncol = 3
  ) +
  labs(
    x = NULL,
    y = "Relative rate scale"
  ) +
  theme_classic() +
  plot_theme1 +
  theme(
    strip.text = element_text(face = "bold"),
    panel.spacing = unit(1.0, "lines")
  )

fig(12, 12)
p_supp_sensitivity


## 6. Supplement 2 — functional response figures

Only the functions retained in the corrected Supplement 2 are included here: Arrhenius temperature response, logistic salinity response, separate current/wave mechanical forcing, a representative Monod saturation response, and the pH Gaussian response.

The obsolete standalone anoxia figure and the old bivariate Teredo Gaussian figure are deliberately omitted.


In [ ]:
arrhenius <- function(T_C, Ea = 50000, T_ref_C = 20) {
  R <- 8.314
  T_K <- T_C + 273.15
  T_ref_K <- T_ref_C + 273.15
  exp((-Ea / R) * ((1 / T_K) - (1 / T_ref_K)))
}

T_seq <- seq(0, 30, length.out = 300)

arr_df <- data.frame(
  temperature = rep(T_seq, 3),
  modifier = c(
    arrhenius(T_seq, Ea = 30000),
    arrhenius(T_seq, Ea = 50000),
    arrhenius(T_seq, Ea = 70000)
  ),
  Ea = factor(
    rep(c("30", "50 baseline", "70"), each = length(T_seq)),
    levels = c("30", "50 baseline", "70")
  )
)

p_arrhenius <- ggplot(
  arr_df,
  aes(temperature, modifier, linetype = Ea, linewidth = Ea)
) +
  geom_line() +
  scale_linetype_manual(
    values = c("30" = "dashed", "50 baseline" = "solid", "70" = "longdash")
  ) +
  scale_linewidth_manual(
    values = c("30" = 0.8, "50 baseline" = 1.2, "70" = 0.8),
    guide = "none"
  ) +
  labs(
    x = "Temperature (°C)",
    y = "Relative modifier",
    linetype = "Activation energy (kJ mol-1)"
  ) +
  geom_hline(yintercept = 1, linetype = "dotted") +
  geom_vline(xintercept = 20, linetype = "dotted") +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_arrhenius


In [ ]:
logistic_salinity <- function(
  S,
  S_mid = 14,
  S_steep = 6,
  f_min = 0.05,
  f_max = 1.0
) {
  f_min + (f_max - f_min) / (1 + exp(-(S - S_mid) / S_steep))
}

S_seq <- seq(0, 40, length.out = 300)

sal_df <- data.frame(
  salinity = rep(S_seq, 5),
  modifier = c(
    logistic_salinity(S_seq, S_mid = 8),
    logistic_salinity(S_seq, S_mid = 10),
    logistic_salinity(S_seq, S_mid = 12),
    logistic_salinity(S_seq, S_mid = 14),
    logistic_salinity(S_seq, S_mid = 16)
  ),
  S_mid = factor(
    rep(c("8", "10", "12", "14 baseline", "16"), each = length(S_seq)),
    levels = c("8", "10", "12", "14 baseline", "16")
  )
)

p_salinity_response <- ggplot(
  sal_df,
  aes(salinity, modifier, linetype = S_mid, linewidth = S_mid)
) +
  geom_line() +
  geom_hline(yintercept = 1, linetype = "dotted") +
  scale_linetype_manual(
    values = c(
      "8" = "dotted",
      "10" = "dotdash",
      "12" = "twodash",
      "14 baseline" = "solid",
      "16" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "8" = 0.8,
      "10" = 0.8,
      "12" = 0.8,
      "14 baseline" = 1.2,
      "16" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Salinity",
    y = "Relative modifier",
    linetype = "Salinity midpoint (PSU)"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_salinity_response


In [ ]:
forcing_modifier <- function(U, U_ref) {
  U / (U + U_ref)
}

U_refs <- c(0.15, 0.30, 0.60)
ref_labels <- c("0.15", "0.30 (baseline)", "0.60")

ref_linetypes <- c(
  "0.15" = "dotted",
  "0.30 (baseline)" = "solid",
  "0.60" = "dashed"
)

ref_linewidths <- c(
  "0.15" = 0.8,
  "0.30 (baseline)" = 1.2,
  "0.60" = 0.8
)

forcing_theme <- theme(
  plot.title = element_text(size = 18, face = "bold", hjust = 0),
  legend.position = "bottom",
  legend.title = element_text(size = 13, face = "bold"),
  legend.text = element_text(size = 12)
)

U <- seq(0, 0.7, length.out = 500)

current_df <- CJ(U = U, U_ref = U_refs)
current_df[, modifier := forcing_modifier(U, U_ref)]
current_df[, U_ref := factor(U_ref, levels = U_refs, labels = ref_labels)]

wave_df <- CJ(U = U, U_ref = U_refs)
wave_df[, modifier := forcing_modifier(U, U_ref)]
wave_df[, U_ref := factor(U_ref, levels = U_refs, labels = ref_labels)]

p_current <- ggplot(
  current_df,
  aes(x = U, y = modifier, linetype = U_ref, linewidth = U_ref)
) +
  geom_line() +
  scale_linetype_manual(values = ref_linetypes, name = "Reference velocity") +
  scale_linewidth_manual(values = ref_linewidths, guide = "none") +
  labs(
    x = expression("Mean current speed (m.s"^{-1}*")"),
    y = "Forcing modifier",
    title = "a) Current forcing"
  ) +
  theme_classic() +
  plot_theme1 +
  forcing_theme

p_wave <- ggplot(
  wave_df,
  aes(x = U, y = modifier, linetype = U_ref, linewidth = U_ref)
) +
  geom_line() +
  scale_linetype_manual(values = ref_linetypes, name = "Reference velocity") +
  scale_linewidth_manual(values = ref_linewidths, guide = "none") +
  labs(
    x = expression("Seabed wave orbital velocity (m.s"^{-1}*")"),
    y = "Forcing modifier",
    title = "b) Wave forcing"
  ) +
  theme_classic() +
  plot_theme1 +
  forcing_theme

shared_legend <- cowplot::get_legend(
  p_current +
    theme(
      legend.position = "bottom",
      legend.box.margin = margin(0, 0, 0, 0)
    )
)

figure_panels <- cowplot::plot_grid(
  p_current + theme(legend.position = "none"),
  p_wave + theme(legend.position = "none"),
  ncol = 2,
  align = "hv",
  axis = "tb"
)

p_mechanical_response <- cowplot::plot_grid(
  figure_panels,
  shared_legend,
  ncol = 1,
  rel_heights = c(1, 0.10)
)

fig(7, 12)
p_mechanical_response


In [ ]:
monod_modifier <- function(x, K = 0.2) {
  x / (x + K)
}

x_seq <- seq(0, 2, length.out = 300)

monod_df <- data.frame(
  x = rep(x_seq, 3),
  modifier = c(
    monod_modifier(x_seq, K = 0.1),
    monod_modifier(x_seq, K = 0.2),
    monod_modifier(x_seq, K = 0.5)
  ),
  K = factor(
    rep(c("0.1", "0.2 (baseline)", "0.5"), each = length(x_seq)),
    levels = c("0.1", "0.2 (baseline)", "0.5")
  )
)

p_monod <- ggplot(
  monod_df,
  aes(x, modifier, linetype = K, linewidth = K)
) +
  geom_line() +
  geom_hline(yintercept = 1, linetype = "dotted") +
  geom_vline(xintercept = 0.2, linetype = "dotted") +
  scale_linetype_manual(
    values = c(
      "0.1" = "dotted",
      "0.2 (baseline)" = "solid",
      "0.5" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "0.1" = 0.8,
      "0.2 (baseline)" = 1.2,
      "0.5" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Resource concentration",
    y = "Relative modifier",
    linetype = "Half-saturation constant"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_monod


In [ ]:
gaussian_modifier <- function(x, x_ref = 8.1, sigma = 0.5) {
  exp(-0.5 * ((x - x_ref) / sigma)^2)
}

pH_seq <- seq(6.5, 9.0, length.out = 300)

ph_df <- data.frame(
  pH = rep(pH_seq, 3),
  modifier = c(
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.25),
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.50),
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.75)
  ),
  sigma = factor(
    rep(c("0.25", "0.50 baseline", "0.75"), each = length(pH_seq)),
    levels = c("0.25", "0.50 baseline", "0.75")
  )
)

p_ph <- ggplot(
  ph_df,
  aes(pH, modifier, linetype = sigma, linewidth = sigma)
) +
  geom_line() +
  geom_hline(yintercept = 1, linetype = "dotted", colour = "grey40") +
  geom_vline(xintercept = 8.1, linetype = "dotted", colour = "grey40") +
  scale_linetype_manual(
    values = c(
      "0.25" = "dotted",
      "0.50 baseline" = "solid",
      "0.75" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "0.25" = 0.8,
      "0.50 baseline" = 1.2,
      "0.75" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "pH",
    y = "Relative biological modifier",
    linetype = "Response width (σ)"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_ph


## 7. Export all manuscript and supplementary figures


In [ ]:
# Manuscript figures
ggsave(
  file.path(figure_out_dir, "regional_time_to_class3.png"),
  threshold_maps,
  width = 12, height = 15, dpi = 300, bg = "white"
)

ggsave(
  file.path(figure_out_dir, "environmental_mechanical_drivers.png"),
  driver_maps,
  width = 12, height = 15, dpi = 300, bg = "white"
)

ggsave(
  file.path(figure_out_dir, "representative_trajectories.png"),
  p_traj,
  width = 12, height = 10, dpi = 300, bg = "white"
)

ggsave(
  file.path(figure_out_dir, "sensitivity_tornado.png"),
  p_sensitivity,
  width = 10, height = 7, dpi = 300, bg = "white"
)

# Supplement 3
ggsave(
  file.path(figure_out_dir, "supp_sensitivity_cell_ratios.png"),
  p_supp_sensitivity,
  width = 12, height = 12, dpi = 300, bg = "white"
)

# Supplement 2 response functions
ggsave(file.path(figure_out_dir, "supp_arrhenius_response.png"),
       p_arrhenius, width = 10, height = 7, dpi = 300, bg = "white")

ggsave(file.path(figure_out_dir, "supp_salinity_response.png"),
       p_salinity_response, width = 10, height = 7, dpi = 300, bg = "white")

ggsave(file.path(figure_out_dir, "supp_mechanical_forcing_response.png"),
       p_mechanical_response, width = 12, height = 7, dpi = 300, bg = "white")

ggsave(file.path(figure_out_dir, "supp_monod_response.png"),
       p_monod, width = 10, height = 7, dpi = 300, bg = "white")

ggsave(file.path(figure_out_dir, "supp_ph_response.png"),
       p_ph, width = 10, height = 7, dpi = 300, bg = "white")

cat("Figures written to:", figure_out_dir, "\n")


## Deliberately left outside this notebook

- Sensitivity-analysis computation/export: keep in the dedicated sensitivity notebook.
- Spatial/model product generation: remains upstream in the production model notebooks.
- Fig. 1 conceptual workflow: not generated in R.
- Exploratory GAM comparisons, diagnostic experiments and legacy plotting code: retained in their development notebooks, not copied here.

This notebook is intended to be boring: run from top to bottom and make the bloody graphs.
